# Final Solution for Polimi RecSys Challenge 2025/26
**Team**: Gorgonzola Racing Team  
**Members**: [Simone Somazzi](https://github.com/SimoSaimon/) & [Filippo Galletta](https://github.com/filippogalletta)  
**Course**: Recommender Systems 2025/26 @ Politecnico di Milano  

## Two-Stage Hybrid Pipeline: Hierarchical Retrieval (M3) & XGBRanker

This notebook contains the complete, reproducible end-to-end pipeline developed by **Gorgonzola Racing Team** for the **Polimi RecSys Challenge 2025/26**.
Our solution achieved an official **Recall@20 of 0.52522** (within 1.2% of the competition winner, outperforming the faculty baseline B5 of 0.51026).

### Architecture Highlights:
* **Stage 1: Candidate Generation (Hierarchical M3 Hybrid)**:
  * Merges similarity matrices of **SLIM ElasticNet** and **EASE_R** ($lpha$).
  * Hierarchically fuses the result with **RP3beta** graph-based similarity ($eta$) into a custom KNN.
  * Linearly blends similarity scores with **Implicit ALS (iALS)** latent factor ratings ($\gamma$).
  * Generates high-quality candidate sets (90 items per user) maximizing the theoretical **Recall Ceiling**.
* **Stage 2: 70+ Feature Engineering & XGBRanker Reranking**:
  * 7 Diverse Base Recommenders (SLIM ElasticNet, EASE_R, RP3beta, P3alpha, iALS, Scaled PureSVD, ItemKNN CF with Tversky similarity).
  * 70+ engineered features capturing:
    * Linf normalized scores, per-user Z-scores (`Score_norm`), rank positions, and inverse log ranks (`RankInv`).
    * Higher-order statistical moments of item similarity to seen profile (Mean, Max, Min, Std Dev, Skewness, Kurtosis).
    * Cross-model consensus and rank distribution (inter-model rank variance, counter recommended, rank mean/std/skewness/kurtosis).
    * Profile statistics and mainstreamness metrics (item popularity, user profile length, mainstream user/item, user profile popularity std, delta popularity).
    * Dual latent factor embeddings (top 5 components from both iALS and ScaledPureSVD).
    * Cross-model comparative features: Score Ratios and Rank Differences between Matrix Factorization and Graph/Linear models.
  * **Pairwise Learning-to-Rank**: XGBRanker trained with pairwise ranking loss (`rank:map` / `rank:pairwise`), tuned across 100+ Bayesian trials via Optuna.
* **Engineering & Memory Management**:
  * Sparse matrix linear algebra (CSR format).
  * Numerical type downcasting via `optimize_df` and proactive garbage collection to handle 2.7M+ candidate pairs.
  * Throughput of ~220 users/sec during inference.


In [ ]:
!pip install --quiet implicit lightfm xgboost optuna


## Environment Setup & Course Framework Integration
We dynamically detect the execution environment (Kaggle, Google Colab, or local machine) and clone the course repository if not already present.


In [ ]:
import os
import sys
import subprocess

# 1. Detect environment and data directory
INPUT_DIR = '.'
if os.path.exists('/kaggle/input/recommender-systems-2025-challenge-polimi'):
    INPUT_DIR = '/kaggle/input/recommender-systems-2025-challenge-polimi'
elif os.path.exists('../data'):
    INPUT_DIR = '../data'
elif os.path.exists('./RecSys_Course_AT_PoliMi'):
    INPUT_DIR = './RecSys_Course_AT_PoliMi'

print(f"Using data directory: {INPUT_DIR}")

# 2. Clone PoliMi course framework if not present
if not os.path.exists('RecSys_Course_AT_PoliMi') and not os.path.exists('../RecSys_Course_AT_PoliMi'):
    print("Cloning RecSys_Course_AT_PoliMi framework...")
    subprocess.run(["git", "clone", "https://github.com/remaplab/RecSys_Course_AT_PoliMi.git"], check=True)

# 3. Add paths to sys.path
for path in ['.', '..', 'RecSys_Course_AT_PoliMi', '../RecSys_Course_AT_PoliMi']:
    abs_path = os.path.abspath(path)
    if abs_path not in sys.path and os.path.exists(abs_path):
        sys.path.append(abs_path)

print("Python search paths updated successfully.")


## Libraries & Module Imports
Import core scientific packages, PoliMi framework modules, and our custom `src/` modules.


In [ ]:
import gc
import time
import numpy as np
import pandas as pd
import scipy.sparse as sps
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from xgboost import XGBRanker, plot_importance

# PoliMi Course Recommenders & Tools
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from Evaluation.Evaluator import EvaluatorHoldout
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender
from Recommenders.GraphBased.P3alphaRecommender import P3alphaRecommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender
from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender

# Gorgonzola Racing Team Custom Modules (src/)
from src.candidate_generation import TripleIntegratedHierarchicalHybridRecommender
from src.features import feature_populator, optimize_df
from src.reranker import XGBoostRerankerRecommender, generate_submission
from src.models import ScaledPureSVDRecommender, FeatureCombinedImplicitALSRecommender

print("All modules imported successfully!")


## Data Loading & Sparse Matrix Construction
Load interaction data and target users, then build the User-Rating Matrix (URM) in CSR format.


In [ ]:
# Locate dataset files
train_csv_path = os.path.join(INPUT_DIR, 'data_train.csv') if os.path.exists(os.path.join(INPUT_DIR, 'data_train.csv')) else 'data_train.csv'
test_csv_path = os.path.join(INPUT_DIR, 'data_target_users_test.csv') if os.path.exists(os.path.join(INPUT_DIR, 'data_target_users_test.csv')) else 'data_target_users_test.csv'

df_train = pd.read_csv(train_csv_path)
df_test_user = pd.read_csv(test_csv_path)

print(f"Loaded {len(df_train):,} interactions and {len(df_test_user):,} target test users.")

# Construct CSR sparse matrix
df_train["row"] = df_train["row"].astype(int)
df_train["col"] = df_train["col"].astype(int)
data = np.ones(len(df_train), dtype=np.float32)

URM_all = sps.csr_matrix((data, (df_train["row"], df_train["col"])))
n_users, n_items = URM_all.shape
density = (URM_all.nnz / (n_users * n_items)) * 100

print(f"URM Shape: {n_users:,} Users x {n_items:,} Items")
print(f"Total Interactions: {URM_all.nnz:,} (Matrix Density: {density:.4f}%)")

del df_train
gc.collect()


## Data Splitting Strategy
To train our two-stage pipeline without data leakage:
1. **Model Training & Validation Split (80%)**: Used to train base recommenders and evaluate candidate generation.
2. **Holdout Test Split (20%)**: Held out to evaluate the end-to-end reranker offline.
3. Within the 80% split, a second holdout split (80/20) is created (`URM_train` and `URM_validation`) to train and calibrate XGBoost before test evaluation.


In [ ]:
# Two-stage Holdout Split
URM_train_validation, URM_test = split_train_in_two_percentage_global_sample(URM_all, train_percentage=0.8)
URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_train_validation, train_percentage=0.8)

evaluator = EvaluatorHoldout(URM_test, cutoff_list=[20])

print(f"URM_all nnz:              {URM_all.nnz:,}")
print(f"URM_train_validation nnz: {URM_train_validation.nnz:,}")
print(f"URM_train nnz:            {URM_train.nnz:,}")
print(f"URM_validation nnz:       {URM_validation.nnz:,}")
print(f"URM_test nnz:             {URM_test.nnz:,}")


## Optimal Hyperparameters for Base Models & M3 Hybrid
Pre-configured with our Bayesian optimization tuning results.


In [ ]:
# Stage 1: M3 Hierarchical Hybrid Weights
best_alpha = 0.15724832635414948   # Similarity merge: (1 - alpha)*SLIM + alpha*EASE
best_beta  = 0.08802471282205815   # Higher-order similarity merge: + beta*RP3beta
best_gamma = 0.12728641243252908   # Score fusion: (1 - gamma)*Similarity + gamma*iALS

# Base Models Hyperparameters
IALS_Parameters = {
    'iterations': 135,
    'factors': 87,
    'alpha': 7.762338288061237,
    'regularization': 0.004799745261257595
}

SLIMElastic_Parameters = {
    'topK': 625,
    'l1_ratio': 0.09517221375634205,
    'alpha': 0.00228698730766055
}

EASE_R_Parameters = {
    'topK': 1431,
    'l2_norm': 426.57622242296605
}

RP3beta_Parameters = {
    'topK': 35,
    'alpha': 0.7733352330682174,
    'beta': 0.4139018623121251,
    'normalize_similarity': True
}

P3alpha_Parameters = {
    'topK': 91,
    'alpha': 0.10032229,
    'normalize_similarity': True
}

ScaledPureSVD_Parameters = {
    'num_factors': 152,
    'scaling_items': 0.000714,
    'scaling_users': 0.575393
}

ItemKNNCF_Parameters = {
    'topK': 8,
    'shrink': 100,
    'tversky_alpha': 0.18445514996044549,
    'tversky_beta': 1.7490566752549062
}

models_to_train = [
    (FeatureCombinedImplicitALSRecommender, IALS_Parameters, "IALS"),
    (SLIMElasticNetRecommender, SLIMElastic_Parameters, "SLIMElastic"),
    (EASE_R_Recommender, EASE_R_Parameters, "EASE_R"),
    (RP3betaRecommender, RP3beta_Parameters, "RP3beta"),
    (P3alphaRecommender, P3alpha_Parameters, "P3alpha"),
    (ScaledPureSVDRecommender, ScaledPureSVD_Parameters, "ScaledPureSVD"),
    (ItemKNNCFRecommender, ItemKNNCF_Parameters, "ItemKNNCF")
]

print(f"Configured {len(models_to_train)} base recommender architectures.")


## Base Models Training on `URM_train` (64% split)
Train all 7 base recommenders on the training split and construct the M3 Candidate Generator.


In [ ]:
other_algorithms = {}

for model_class, params, name in models_to_train:
    print(f"Training {name}...")
    recommender = model_class(URM_train)
    recommender.fit(**params)
    other_algorithms[name] = recommender

# Stage 1: Build M3 Hierarchical Hybrid
print("\nFitting TripleIntegratedHierarchicalHybridRecommender (M3)... ")
linear_comb_rec = TripleIntegratedHierarchicalHybridRecommender(
    URM_train, 
    other_algorithms['SLIMElastic'], 
    other_algorithms['EASE_R'], 
    other_algorithms['RP3beta'], 
    other_algorithms['IALS']
)
linear_comb_rec.fit(best_alpha, best_beta, best_gamma)
print("M3 Candidate Generator ready!")


## Stage 1: Candidate Generation & Feature Engineering
Generate top 90 candidates per user using our M3 model, compute the 70+ tabular features via `src/features.py`, and attach ground-truth labels from `URM_validation`.


In [ ]:
candidates_cutoff = 90

print(f"Populating features for {candidates_cutoff} candidates per user...")
training_dataframe = feature_populator(
    URM_train, 
    linear_comb_rec, 
    other_algorithms, 
    cutoff=candidates_cutoff
)

training_dataframe = optimize_df(training_dataframe)

# Attach ground-truth labels from URM_validation
URM_val_coo = sps.coo_matrix(URM_validation)
correct_recs = pd.DataFrame({
    "UserID": URM_val_coo.row,
    "ItemID": URM_val_coo.col
})

training_dataframe = pd.merge(
    training_dataframe, 
    correct_recs, 
    on=['UserID', 'ItemID'], 
    how='left', 
    indicator='Exist'
)
training_dataframe["Label"] = (training_dataframe["Exist"] == "both").astype(int)
training_dataframe.drop(columns=['Exist'], inplace=True)

# Verification of Recall Ceiling
captured = training_dataframe['Label'].sum()
total_val_positives = URM_validation.nnz
recall_ceiling = captured / total_val_positives

print("\n--- RECALL CEILING VERIFICATION ---")
print(f"Candidates cutoff:          {candidates_cutoff}")
print(f"Positives captured in pool: {captured:,}")
print(f"Total validation positives: {total_val_positives:,}")
print(f"THEORETICAL RECALL CEILING: {recall_ceiling:.4f}")
print("-----------------------------------")


## Stage 2: XGBoost Reranking Model
We instantiate `XGBRanker` with the optimal hyperparameters identified via 100+ Bayesian optimization trials using Optuna (Trial 96).


In [ ]:
XGBoost_Parameters = {    
    'objective': 'rank:map', 
    'n_estimators': 2261, 
    'learning_rate': 0.007702162822426045, 
    'reg_alpha': 0.00991022085129717, 
    'reg_lambda': 0.00014710271979317914, 
    'max_depth': 7, 
    'max_leaves': 349, 
    'grow_policy': 'lossguide', 
    'gamma': 0.3793589358094316, 
    'min_child_weight': 0.002662290661537342, 
    'subsample': 0.6587292706752251, 
    'colsample_bytree': 0.48594318797001984,
    'tree_method': 'hist',
    'verbosity': 1
}

# Group sizes per user
groups = training_dataframe.groupby("UserID").size().values
y_train = training_dataframe["Label"]
feature_cols = [c for c in training_dataframe.columns if c not in ['UserID', 'ItemID', 'Label']]
X_train = training_dataframe[feature_cols]

print(f"Training XGBRanker on {len(X_train):,} rows and {len(feature_cols)} features across {len(groups):,} query groups...")
XGB_model = XGBRanker(**XGBoost_Parameters)
XGB_model.fit(X_train, y_train, group=groups, verbose=True)
print("XGBoost training completed!")


## Feature Importance Analysis
Analyze which features contribute most to the reranking performance (Gain / Weight).


In [ ]:
try:
    importance_dict = XGB_model.get_score(importance_type='weight')
except AttributeError:
    importance_dict = XGB_model.get_booster().get_score(importance_type='weight')

importance_df = pd.DataFrame({
    'Feature': list(importance_dict.keys()),
    'Importance': list(importance_dict.values())
}).sort_values(by='Importance', ascending=False)

print("Top 20 Most Important Features:")
print(importance_df.head(20).to_string(index=False))

# Plot feature importance
fig, ax = plt.subplots(figsize=(10, 14))
plot_importance(XGB_model, importance_type='weight', max_num_features=30, ax=ax, title='Top 30 Feature Importance (Weight)')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300)
plt.show()


## Validation on Holdout Test Split (20%)
To validate offline performance, we train the base models on `URM_train_validation` (80%), generate validation features, and evaluate with `EvaluatorHoldout`.


In [ ]:
print("Training base models on URM_train_validation (80%)...")
other_algorithms_f = {}

for model_class, params, name in models_to_train:
    recommender = model_class(URM_train_validation)
    recommender.fit(**params)
    other_algorithms_f[name] = recommender

linear_comb_rec_f = TripleIntegratedHierarchicalHybridRecommender(
    URM_train_validation, 
    other_algorithms_f['SLIMElastic'], 
    other_algorithms_f['EASE_R'], 
    other_algorithms_f['RP3beta'], 
    other_algorithms_f['IALS']
)
linear_comb_rec_f.fit(best_alpha, best_beta, best_gamma)

print("Generating validation feature dataset...")
val_dataframe = feature_populator(
    URM_train_validation, 
    linear_comb_rec_f, 
    other_algorithms_f, 
    cutoff=candidates_cutoff
)
val_dataframe = optimize_df(val_dataframe)

# Create Reranker Recommender wrapper and evaluate
reranker_eval = XGBoostRerankerRecommender(URM_train_validation, XGB_model, val_dataframe)
result_df, _ = evaluator.evaluateRecommender(reranker_eval)

print("\n--- OFFLINE HOLDOUT EVALUATION RESULTS ---")
print(result_df)
print("------------------------------------------")


## Full Retraining on 100% Data (`URM_all`) & Final Submission
Finally, we retrain the base models and M3 hybrid on the entire interaction dataset (`URM_all`), generate prediction features for the target test users, and export the official submission file.


In [ ]:
print("Training base models on 100% data (URM_all)...")
other_algorithms_all = {}

for model_class, params, name in models_to_train:
    print(f"Training {name} on URM_all...")
    recommender = model_class(URM_all)
    recommender.fit(**params)
    other_algorithms_all[name] = recommender

linear_comb_rec_all = TripleIntegratedHierarchicalHybridRecommender(
    URM_all, 
    other_algorithms_all['SLIMElastic'], 
    other_algorithms_all['EASE_R'], 
    other_algorithms_all['RP3beta'], 
    other_algorithms_all['IALS']
)
linear_comb_rec_all.fit(best_alpha, best_beta, best_gamma)

print("\nGenerating prediction feature dataset for URM_all...")
prediction_dataframe = feature_populator(
    URM_all, 
    linear_comb_rec_all, 
    other_algorithms_all, 
    cutoff=candidates_cutoff
)
prediction_dataframe = optimize_df(prediction_dataframe)

# Final Reranker instance
final_recommender = XGBoostRerankerRecommender(URM_all, XGB_model, prediction_dataframe)

# Generate CSV submission
target_users = df_test_user["user_id"].tolist()
df_submission = generate_submission(
    final_recommender, 
    target_users, 
    output_path="recommendations_xgboost.csv", 
    cutoff=20
)

print("\nFirst 5 generated recommendations:")
print(df_submission.head())


## Conclusion & Pipeline Summary
* **Two-Stage Synergy**: The M3 Hierarchical Hybrid guarantees high candidate coverage with a compact, manageable cutoff (90 items), while XGBRanker leverages 70+ statistical and embedding features to achieve precise pairwise ordering.
* **Result**: This architecture achieved **Recall@20 = 0.52522** on the official competition leaderboard.
* **Credits**: Built on the framework developed by [Maurizio Ferrari Dacrema](https://github.com/remaplab/RecSys_Course_AT_PoliMi) for the Recommender Systems course @ Politecnico di Milano.
